# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UmairMehfooz/Ml-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. My Data Contract

**What one row means:**  
One row represents the daily performance of one content item for one client on one report date. The grain is `client_hash_id × content_hash_id × report_date`.

**Table:**  
I will use `fact_content_daily_performance` as the main table for this lane.

**Time window:**  
I will use March 2026 as the development month. I will not use the `_sample` table for developing label logic because it represents June 2026, the final month, which should be treated as a sealed outcome/test period.

**What I predict:**  
My lane is Content Refresh Prioritization. I want to predict whether a content item is likely to experience a future decline in search performance, so that the SEO team can prioritize which content should be refreshed.

**What I deliberately exclude:**  
I will exclude future-derived performance variables and label-derived fields from the features because they would reveal information that would not be available at the decision moment.

In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q duckdb huggingface_hub

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features:
impressions
clicks
CTR
average position
content age
search volume

Label:
Future Decline

Context:
content_id
client_id
report_date



In [24]:
!pip install -q duckdb

import duckdb

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

print("DuckDB connected to Hugging Face.")

con.sql(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    """
)

march_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-03/**/*.parquet"
)

print(march_path)

con.sql(
    f"""
    SELECT COUNT(*) AS march_rows
    FROM read_parquet('{march_path}')
    """
)

con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
    LIMIT 1
    """
)

DuckDB connected to Hugging Face.
hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet


┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [25]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,

    gsc_impressions,
    gsc_clicks,

    CASE
        WHEN gsc_impressions > 0
        THEN gsc_clicks * 1.0 / gsc_impressions
        ELSE NULL
    END AS gsc_ctr,

    CASE
        WHEN gsc_impressions > 0
        THEN gsc_sum_position * 1.0 / gsc_impressions
        ELSE NULL
    END AS gsc_avg_position,

    sessions_organic

FROM read_parquet('{march_path}')
LIMIT 100
"""

feature_frame = con.sql(feature_query)

feature_frame

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬───────────────────────┬────────────────────┬──────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │        gsc_ctr        │  gsc_avg_position  │ sessions_organic │
│         varchar         │         varchar          │    date     │      int64      │   int64    │        double         │       double       │      int64       │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼───────────────────────┼────────────────────┼──────────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │              20 │          0 │                   0.0 │               3.35 │             NULL │
│ client_73cda7b4e4f265ea │ content_05597932fe4da067 │ 2026-03-01  │               1 │          0 │                   0.0 │                0.0 │             NULL │
│ client_73cda7b

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification 1 — Grain

I expect the combination of `client_hash_id`, `content_hash_id`, and `report_date` to uniquely identify a row. I therefore check for duplicate combinations. If the query returns zero rows, there are no duplicate grain combinations in the March slice.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_query = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    sessions_organic,
    ga4_engaged_sessions
FROM read_parquet('{march_path}')
LIMIT 100
"""

feature_frame = con.sql(feature_query)

feature_frame

┌─────────────────────────┬──────────────────────────┬─────────────┬─────────────────┬────────────┬────────────────────┬──────────────────┬──────────────────────┐
│     client_hash_id      │     content_hash_id      │ report_date │ gsc_impressions │ gsc_clicks │  gsc_avg_position  │ sessions_organic │ ga4_engaged_sessions │
│         varchar         │         varchar          │    date     │      int64      │   int64    │       double       │      int64       │        int64         │
├─────────────────────────┼──────────────────────────┼─────────────┼─────────────────┼────────────┼────────────────────┼──────────────────┼──────────────────────┤
│ client_73cda7b4e4f265ea │ content_b7e512995f79d5a6 │ 2026-03-01  │              20 │          0 │               3.35 │             NULL │                 NULL │
│ client_73cda7b4e4f265ea │ content_05597932fe4da067 │ 2026-03-01  │               1 │          0 │                0.0 │             NULL │                 NULL │
│ client_73cda7b4e4f26

### Verification 2 — Row count and date span

I check the number of rows in the March 2026 slice and verify the earliest and latest report dates. This confirms that the slice contains the time period I intended to use.

In [27]:
date_query = f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{march_path}')
"""

date_check = con.sql(date_query)

date_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────┬────────────┐
│ row_count │ first_date │ last_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

In [28]:
availability_query = f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
"""

availability_check = con.sql(availability_query)

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ available_rows │
│     int64      │
├────────────────┤
│         364347 │
└────────────────┘

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(
    f"""
    SELECT *
    FROM read_parquet('{march_path}')
    LIMIT 5
    """
)

┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │ sessions_ai │ ai_chatgpt │ ai_perplexity │ ai_gemini │ ai_copilot │ ai_clau

In [30]:
march_features_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(sessions_organic) AS sessions_organic,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

FROM read_parquet('{march_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

march_features = con.sql(march_features_query).df()

march_features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,sessions_organic,ga4_engaged_sessions
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.145765,NaN,NaN
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,4.909314,NaN,NaN
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,0.0,5.177774,NaN,NaN
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,1.0,4.685335,NaN,NaN
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0,0.0,4.266667,NaN,NaN


In [31]:
april_path = (
    f"{rel}/fact_content_daily_performance/"
    "month=2026-04/**/*.parquet"
)

print(april_path)

hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/**/*.parquet


In [32]:
april_outcome_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions) AS april_impressions

FROM read_parquet('{april_path}')

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
"""

april_outcome = con.sql(april_outcome_query).df()

april_outcome.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,april_impressions
0,client_62f4a7e64f5e0096,content_143987cfdeaba4c0,187.0
1,client_62f4a7e64f5e0096,content_13a8105125458098,23.0
2,client_62f4a7e64f5e0096,content_e2bd76be7eed690d,17.0
3,client_62f4a7e64f5e0096,content_86ab16840c4e0e1a,202.0
4,client_62f4a7e64f5e0096,content_3f87f49c36774e23,93.0


In [33]:
import pandas as pd

model_frame = march_features.merge(
    april_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_frame.head()

model_frame["impression_change_pct"] = (
    (model_frame["april_impressions"] - model_frame["gsc_impressions"])
    / model_frame["gsc_impressions"]
) * 100

model_frame["decline_label"] = (
    model_frame["impression_change_pct"] < -20
).astype(int)


model_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "april_impressions",
        "impression_change_pct",
        "decline_label"
    ]
].head(10)


,client_hash_id,content_hash_id,gsc_impressions,april_impressions,impression_change_pct,decline_label
0,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1027.0,14.238042,0
1,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,12.0,-64.705882,1
2,client_62f4a7e64f5e0096,content_d49a012dcb924e31,329.0,109.0,-66.869301,1
3,client_62f4a7e64f5e0096,content_614baf2af4330bd7,772.0,278.0,-63.989637,1
4,client_62f4a7e64f5e0096,content_4a1ca0fa5c177e0c,14.0,6.0,-57.142857,1
5,client_62f4a7e64f5e0096,content_225dc9235023be5f,488.0,143.0,-70.696721,1
6,client_62f4a7e64f5e0096,content_babcf791dccc1610,181.0,66.0,-63.535912,1
7,client_62f4a7e64f5e0096,content_26f5092ee7f70d45,4314.0,1548.0,-64.116829,1
8,client_62f4a7e64f5e0096,content_cfad137c1b04251b,438.0,194.0,-55.707763,1
9,client_62f4a7e64f5e0096,content_9e7c70abfbae371e,2436.0,578.0,-76.272578,1


In [34]:
model_frame["decline_label"].value_counts(normalize=True)

,proportion
decline_label,
0,0.521845
1,0.478155


In [35]:
model_frame["decline_label"].value_counts()

,count
decline_label,
0,82738
1,75811


In [36]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model_frame["LEAKED_FUTURE_LABEL"] = model_frame["decline_label"]

In [37]:
feature_columns_leaky = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "ga4_engaged_sessions",
    "LEAKED_FUTURE_LABEL"
]

X = model_frame[feature_columns_leaky]
y = model_frame["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

leaky_accuracy = accuracy_score(y_test, predictions)

print("Accuracy with leakage:", leaky_accuracy)

Accuracy with leakage: 1.0


In [38]:
model_frame = model_frame.drop(
    columns=["LEAKED_FUTURE_LABEL"]
)

In [39]:
feature_columns_honest = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "sessions_organic",
    "ga4_engaged_sessions"
]

X = model_frame[feature_columns_honest]
y = model_frame["decline_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

honest_accuracy = accuracy_score(y_test, predictions)

print("Honest accuracy:", honest_accuracy)

Honest accuracy: 0.5757174392935982


## The leakage trap

I deliberately added `LEAKED_FUTURE_LABEL` as a feature. This column was calculated from the future April outcome, so it would not be available when the March refresh decision was made.

The model's score became extremely high when this leaked column was included. This does not mean the model is good. It means the model was given information that directly contains the answer.

After removing the leaked column, the score dropped to the honest result.

This demonstrates why a feature must be available at the decision moment, not merely correlated with the target.

The final feature set therefore excludes all future-derived outcome variables.

## 4. Limitation

A key limitation is that the warehouse is an unbalanced panel: different clients have different amounts of historical data. Therefore, using March 2026 as a global development month does not guarantee that every client has the same amount of prior history.

I also require `gsc_data_available IS TRUE` when constructing the search-performance features and outcome. Missing availability should not be interpreted as zero performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.